# TACO v1.3 Trace Dataset EDA

We start the exploratory analysis for the v1.3 trace datasets by focusing on a single shard. The numbers produced
here do not cover the whole dataset, but since the shards are a good random sample of the overall dataset, the
results should be somewhat representative.

We measure how much of the original dataset survived, review trace tags, quantify augmentation, etc.; the contents of
this notebooks are not meant to be used as demonstrations of anything, so no, it is not really documented...

In [ ]:
import collections

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display

import pyine.data.taco.dataset_utils
import pyine.data.traces.dataset_reader
import pyine.data.traces.dataset_utils
import pyine.data.utils.splits
import pyine.utils.filesystem

sns.set_theme(style="whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "axes.titlesize": 14,
        "axes.labelsize": 12,
    }
)

In [ ]:
# ------------ CHANGE THESE SETTINGS IF NEEDED ------------
source_dataset_name = "TACO"  # this notebook targets the TACO v1.3 10s10t traces dataset
source_dataset_path = pyine.data.taco.dataset_utils.get_latest_repackaged_dataset_path()
trace_dataset_pattern = "v1.3/10s10t.*of000026.*.lmdb"  # specific pattern for v1.3 of the dataset
selected_part_idx = 0  # zero-based index; tweak to inspect other individual shards
expected_part_count = 26  # given the 500-problem-chunk split used for the v1.3 dataset
expected_solution_count_per_problem = 10  # the '10s' part of the v1.3 10s10t dataset
expected_test_count_per_problem = 10  # the '10t' part of the v1.3 10s10t dataset
validate_all_shards_overlap = True
# ---------------------------------------------------------

In [ ]:
# fetch the dataset paths, split data, and parts from the framework utility functions
dataset_paths = sorted(
    pyine.data.traces.dataset_utils.get_matching_dataset_paths(
        source_dataset_name=source_dataset_name,
        pattern=trace_dataset_pattern,
    )
)
part_file_paths = pyine.data.utils.splits.get_dataset_split_part_file_paths(source_dataset_name)
assert len(dataset_paths) == len(part_file_paths), "dataset shards and split shard files diverge"
assert len(dataset_paths) == expected_part_count, "unexpected number of dataset shards"

# inspect the split and make sure that it truly was used to prepare the traces dataset
split = pyine.data.utils.splits.get_dataset_split_result(source_dataset_name)
all_problem_ids: list[str] = []
part_problem_ids: list[list[str]] = []
for part_file_path in part_file_paths:
    with part_file_path.open("r") as fd:
        target_problem_ids = yaml.safe_load(fd)
    part_problem_ids.append(target_problem_ids)
    assert not any([pid in all_problem_ids for pid in target_problem_ids]), "problem id already seen?"
    all_problem_ids.extend(target_problem_ids)
print(f"source dataset metadata parsed: expecting {len(all_problem_ids)} problems over {len(dataset_paths)} shards")
assert set(all_problem_ids) == set(split.identifiers), "unexpected problem ids found?"
assert set(all_problem_ids) == set(split.subset_assignments.keys())

if validate_all_shards_overlap:
    traced_problem_ids: list[str] = []
    for dataset_path, expected_problem_ids in zip(dataset_paths, part_problem_ids):
        print(f"validating: {dataset_path}")
        try:
            reader = pyine.data.traces.dataset_reader.DatasetReader(dataset_path)
            found_problem_ids = [problem_id for problem_id in expected_problem_ids if problem_id in reader.problem_keys]
            print(f"\tfound problems ratio: {len(found_problem_ids) / len(expected_problem_ids):.2f}")
            unexpected_problem_ids = [
                problem_id for problem_id in reader.problem_keys if problem_id not in expected_problem_ids
            ]
            print(f"\tunexpected problems: {len(unexpected_problem_ids)}")
            assert not any([pid in traced_problem_ids for pid in found_problem_ids]), "problem id already seen?"
            traced_problem_ids.extend(found_problem_ids)
        except Exception as e:
            print(f"\tfailed: {e}")
    assert len(traced_problem_ids) == len(set(traced_problem_ids)), "duplicate problem ids found?"
    assert set(traced_problem_ids).issubset(set(all_problem_ids)), "unexpected problem ids found?"

selected_dataset_path = dataset_paths[selected_part_idx]
with part_file_paths[selected_part_idx].open() as fd:
    expected_problem_ids = yaml.safe_load(fd)
reader = pyine.data.traces.dataset_reader.DatasetReader(selected_dataset_path)
assert len(reader) > 0, "no traces found in selected shard"
available_problem_ids = reader.problem_keys
print(f"selected dataset shard: {selected_dataset_path.resolve()}")
print(f"expected problem ids from split: {len(expected_problem_ids)}")
print(f"problems present in shard: {len(available_problem_ids)}")

In [ ]:
target_problem_ids = sorted(set(expected_problem_ids).union(available_problem_ids))
problem_iterator = pyine.data.traces.dataset_utils.CodingProblemIterator(
    dataset_name=source_dataset_name,
    root_data_path=source_dataset_path,
    target_problem_ids=target_problem_ids,
    reformat_code_strings=False,
    validate_code_strings=False,
    show_progress=False,
)
original_problem_summary: dict[str, dict[str, object]] = {}
for problem, solutions in problem_iterator:
    problem_key = str(problem.problem_id)
    solution_ids = {str(solution.solution_id) for solution in solutions}
    original_problem_summary[problem_key] = {
        "solution_ids": solution_ids,
        "test_count": len(problem.test_inout_pairs),
        "problem_tags": problem.problem_tags,
    }
missing_original_metadata = sorted(
    {problem_id for problem_id in target_problem_ids if problem_id not in original_problem_summary}
)
if missing_original_metadata:
    print(f"missing metadata for {len(missing_original_metadata)} unexpected problems")
print(f"parsed metadata for {len(original_problem_summary)} problems from the source dataset")

In [ ]:
trace_tags_counter = collections.Counter()
augment_categories_counter = collections.Counter()
problem_trace_counts = collections.Counter()
problem_aug_trace_counts = collections.Counter()
problem_children_map: dict[str, set[str]] = collections.defaultdict(set)
problem_test_idxs: dict[str, set[int]] = collections.defaultdict(set)
problem_augmented_categories: dict[str, collections.Counter] = {}
solution_trace_counts = collections.Counter()
solution_aug_counts = collections.Counter()
solution_test_idxs: dict[str, set[int]] = collections.defaultdict(set)
solution_parents_map: dict[str, str] = {}

for trace_key, trace_tags in zip(reader.trace_keys, reader.trace_tag_lists):
    trace_identifier = pyine.data.traces.dataset_utils.TraceIdentifier.from_string(trace_key)
    solution_identifier = trace_identifier.get_parent_identifier()
    problem_identifier = solution_identifier.get_parent_identifier()
    problem_id_str = str(problem_identifier)
    solution_id_str = str(solution_identifier)
    is_augmented = trace_identifier.augment_category is not None
    if solution_id_str not in solution_parents_map:
        solution_parents_map[solution_id_str] = problem_id_str
    else:
        assert solution_parents_map[solution_id_str] == problem_id_str
    problem_children_map[problem_id_str].add(solution_id_str)
    problem_trace_counts[problem_id_str] += 1
    solution_trace_counts[solution_id_str] += 1
    if is_augmented:
        problem_aug_trace_counts[problem_id_str] += 1
        solution_aug_counts[solution_id_str] += 1
        augment_categories_counter[trace_identifier.augment_category] += 1
        if problem_id_str not in problem_augmented_categories:
            problem_augmented_categories[problem_id_str] = collections.Counter()
        problem_augmented_categories[problem_id_str][trace_identifier.augment_category] += 1
    problem_test_idxs[problem_id_str].add(trace_identifier.test_idx)
    solution_test_idxs[solution_id_str].add(trace_identifier.test_idx)
    trace_tags_counter.update(trace_tags)

missing_parent_keys = [
    aug_key
    for aug_key, parent_key in reader.augment_key_to_parent_trace_key.items()
    if parent_key not in reader.trace_keys
]
parent_categories = {
    pyine.data.traces.dataset_utils.TraceIdentifier.from_string(parent_key).augment_category
    for parent_key in reader.augment_key_to_parent_trace_key.values()
}
assert not missing_parent_keys, "augmented traces without parent augmentless trace detected"
assert parent_categories == {None}, "found parents that were themselves augmented"

print(f"parsed metadata for {len(reader)} traces from the selected shard")

In [ ]:
all_problem_ids = sorted(set(expected_problem_ids).union(available_problem_ids))
problem_rows: list[dict[str, object]] = []
for problem_id in all_problem_ids:
    original_entry = original_problem_summary.get(problem_id, {})
    original_solution_ids: set[str] = original_entry.get("solution_ids", set())
    original_solution_count = len(original_solution_ids)
    original_test_count = original_entry.get("test_count", 0)
    problem_tags = original_entry.get("problem_tags", [])
    found_solution_count = len(problem_children_map.get(problem_id, set()))
    found_test_count = len(problem_test_idxs.get(problem_id, set()))
    found_traces_total = problem_trace_counts.get(problem_id, 0)
    found_traces_augmented = problem_aug_trace_counts.get(problem_id, 0)
    found_augment_categories = problem_augmented_categories.get(problem_id, collections.Counter())
    row = dict(
        problem_id=problem_id,
        is_expected=problem_id in expected_problem_ids,
        is_present=found_traces_total > 0,
        orig_problem_tags=", ".join(sorted(problem_tags)),
        original_solution_count=original_solution_count,
        original_test_count=original_test_count,
        max_possible_traces_non_aug=original_solution_count * original_test_count,
        found_traces_non_aug=found_traces_total - found_traces_augmented,
        found_traces_aug=found_traces_augmented,
        found_traces_total=found_traces_total,
        found_solution_count=found_solution_count,
        found_test_count=found_test_count,
        found_augment_categories=", ".join(f"{name}:{count}" for name, count in found_augment_categories.most_common()),
        found_solution_ratio=(found_solution_count / original_solution_count if original_solution_count else np.nan),
        found_solution_expected_ratio=(
            found_solution_count / min(expected_solution_count_per_problem, original_solution_count)
        ),
        found_test_ratio=(found_test_count / original_test_count if original_test_count else np.nan),
        found_test_expected_ratio=(found_test_count / min(expected_test_count_per_problem, original_test_count)),
    )
    problem_rows.append(row)

problem_df = pd.DataFrame(problem_rows)
present_problem_df = problem_df[problem_df["is_present"]].copy()
expected_problem_df = problem_df[problem_df["is_expected"]].copy()

print(f"problems dataframe has {len(problem_df)} entries (first 10 shown below):")
display(problem_df.head(n=10))

print(f"present problems dataframe has {len(present_problem_df)} entries (first 10 shown below):")
display(present_problem_df.head(n=10))

In [ ]:
original_solution_total_present = present_problem_df["original_solution_count"].sum()
found_solution_total_present = present_problem_df["found_solution_count"].sum()
original_test_total_present = present_problem_df["original_test_count"].sum()
found_test_total_present = present_problem_df["found_test_count"].sum()
found_traces_total_present = present_problem_df["found_traces_total"].sum()
assert found_traces_total_present == len(reader), "inconsistent trace counts"
max_possible_traces_present = present_problem_df["max_possible_traces_non_aug"].sum()
found_non_aug_trace_total_present = present_problem_df["found_traces_non_aug"].sum()
found_aug_trace_total_present = present_problem_df["found_traces_aug"].sum()
assert found_non_aug_trace_total_present == found_traces_total_present - found_aug_trace_total_present
total_augmented_traces = len(reader.augment_key_to_parent_trace_key)
assert found_aug_trace_total_present == total_augmented_traces
total_solutions = len(solution_parents_map)
augmented_types_count = collections.defaultdict(int)
for counts_str in present_problem_df["found_augment_categories"].to_list():
    augm_counts = counts_str.split(", ") if counts_str else []
    for augm_count in augm_counts:
        augm_category, count = augm_count.split(":")
        augmented_types_count[augm_category] += int(count)
assert sum(augmented_types_count.values()) == total_augmented_traces, "inconsistent augmented trace counts"
augmented_types_share = {
    augm_category: count / total_augmented_traces for augm_category, count in augmented_types_count.items()
}
summary_metrics = pd.Series(
    dict(
        dataset_file_name=selected_dataset_path.name,
        expected_problem_count=len(expected_problem_ids),
        problem_count_in_dataset=len(available_problem_ids),
        expected_problem_coverage_ratio=float(expected_problem_df["is_present"].mean()),
        unexpected_problem_count=int(present_problem_df.loc[~present_problem_df["is_expected"], "is_present"].sum()),
        solution_retention_ratio_on_present_problems=found_solution_total_present / original_solution_total_present,
        solution_retention_ratio_for_expectation=float(expected_problem_df["found_solution_expected_ratio"].mean()),
        test_retention_ratio_on_present_problems=found_test_total_present / original_test_total_present,
        test_retention_ratio_for_expectation=float(expected_problem_df["found_test_expected_ratio"].mean()),
        trace_count=found_traces_total_present,
        non_augm_trace_count=found_non_aug_trace_total_present,
        non_augm_trace_share=found_non_aug_trace_total_present / found_traces_total_present,
        non_augm_trace_dataset_coverage_ratio=found_non_aug_trace_total_present / max_possible_traces_present,
        avg_non_augm_traces_per_problem=float(found_non_aug_trace_total_present / len(available_problem_ids)),
        avg_non_augm_traces_per_solution=float(found_non_aug_trace_total_present / total_solutions),
        augm_trace_count=found_aug_trace_total_present,
        augm_trace_share=found_aug_trace_total_present / found_traces_total_present,
        augmented_types_share=", ".join(
            f"{augm_category}:{share}" for augm_category, share in augmented_types_share.items()
        ),
        avg_augm_traces_per_problem=float(total_augmented_traces / len(available_problem_ids)),
        avg_augm_traces_per_solution=float(total_augmented_traces / total_solutions),
    )
)

print("shard dataset summary statistics:")
display(summary_metrics.to_frame(name="value"))

In [ ]:
tag_df = pd.DataFrame(
    {
        "tag": list(trace_tags_counter.keys()),
        "count": list(trace_tags_counter.values()),
    }
).sort_values("count", ascending=False)
tag_df["share"] = tag_df["count"] / tag_df["count"].sum()
print("top-25 tags across all traces:")
display(tag_df.head(25))

augment_category_df = pd.DataFrame(
    {
        "augment_category": list(augment_categories_counter.keys()),
        "count": list(augment_categories_counter.values()),
    }
).sort_values("count", ascending=False)
augment_category_df["share"] = augment_category_df["count"] / augment_category_df["count"].sum()
print("augmentation categories:")
display(augment_category_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(
    data=present_problem_df,
    x="found_solution_ratio",
    bins=15,
    ax=axes[0],
)
axes[0].set_title("Solution retention ratio per problem")
axes[0].set_xlabel("new / original solutions")
sns.histplot(
    data=present_problem_df,
    x="found_test_ratio",
    bins=15,
    ax=axes[1],
)
axes[1].set_title("Test retention ratio per problem")
axes[1].set_xlabel("new / original tests")
plt.suptitle("Retention Distributions", y=1.04)
plt.tight_layout()

plt.figure(figsize=(10, 6))
plot_tags = tag_df.head(25).iloc[::-1]
sns.barplot(data=plot_tags, x="count", y="tag", color="C0")
plt.title("Top 25 Trace Tags (all traces)")
plt.xlabel("trace count")
plt.ylabel("tag")
plt.tight_layout()

In [ ]:
top_problem_shortfall = expected_problem_df.loc[
    expected_problem_df["is_present"].eq(False),
    ["problem_id"],
]
print("Examples of expected problems missing from shard:")
display(top_problem_shortfall.head(10))

## Summary of findings

Single shard analysis: `v1.3/10s10t.000001of000026.2025-09-17.lmdb`, 2025-09-23
- 63.2% of the 500 expected problems are present in this shard (316/500); there are no unexpected problems in the LMDB.
- Across the 316 present problems, we retain ~6.6% of original solutions (2,293/34,496) and ~9.3% of original tests (2,155/23,240); augmentless execution coverage is ~0.35% of the theoretical maximum (14,839/4,234,362).
- Augmented traces make up ~48.3% of the shard (13,875/28,714), averaging ~6.1 augmentations per solution; all augmentations are tagged `obfuscated`, with validated parent traces in place.
- Tags indicate most traces are from Codeforces and carry some execution metadata (e.g., `return:has_stdout`, step counts). Difficulty tags show substantial presence of `MEDIUM`.

Full analysis (all 26 v1.3 shards), TODO!
- TODO!